# Kubernetes Core Objects

A hands-on tour of the core Kubernetes objects on a local k3d cluster (1 server, 3 agents).

- **Cluster basics**: what Kubernetes solves (availability, scalability), control plane vs worker nodes, inspecting clusters and contexts, creating a k3d cluster
- **Namespaces**: listing, creating and switching the default namespace for `kubectl`
- **Workloads**: Pods and Deployments (from the CLI and from manifests), scaling, self-healing, and inspecting objects with `describe`
- **Labels**: viewing, selecting (`-l`) and adding labels to resources
- **Networking**: the data plane, port forwarding, Services (ClusterIP, LoadBalancer), Endpoints and CoreDNS, Ingress and the ingress controller with a local hostname (`k8s.local`)
- **Configurations**: ConfigMaps (mounted as a volume and injected as environment variables), with Secrets covered conceptually
- **Storage**: StorageClass, PersistentVolume and PersistentVolumeClaim, dynamic (`local-path`) vs static (`manual`) provisioning, data surviving pod deletion, and the `Delete` vs `Retain` reclaim policies
- **Cleanup**: removing the storage objects, deleting the cluster, reclaiming Docker disk space and removing the `k8s.local` host entry


### Seed question

*Should your application server crash at 2am in the morning, how long does it take before you know your workloads aren't running anymore? how long does it take to get them back online?*


## Why use Kubernetes?

- **Availability**: when an app or node crashes, Kubernetes restarts or reschedules it (self-healing)
- **Scalability**: scale replicas up or down, and load-balance traffic across them
- **Disaster recovery**: because the desired state is declared, a lost environment can be rebuilt by re-applying it. Backing up the data itself is still your job.

## The Cluster Architecture 

### The Control Plane 

The control plane holds and reconciles your declared <u>intent</u>. 

![control](images/control%20plane.gif)

### Get connected clusters

In [ ]:
%%bash
kubectl config get-clusters

In [ ]:
%%bash
kubectl config get-contexts

### Create a cluster

- We create a demo cluster with 1 control plane and 3 worker nodes using k3d.
- k3d is a lightweight wrapper that runs k3s (a minimal Kubernetes distribution) inside Docker containers, so you can spin up local multi-node Kubernetes clusters in seconds.

![k3d](images/k3d%20cluster.gif)

In [ ]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
if k3d cluster list objectscluster >/dev/null 2>&1; then
  echo "Cluster 'objectscluster' already exists, skipping creation."
else
  k3d cluster create objectscluster \
    --image ghcr.io/k3s-io/k3s:v1.35.3-k3s1 \
    --servers 1 \
    --agents 3
fi

In [ ]:
%%bash
kubectl config get-contexts

In [ ]:
%%bash
kubectl get nodes
echo ""
kubectl get nodes --context rancher-desktop

In [ ]:
%%bash
kubectl config view --minify --context=k3d-objectscluster

## Namespaces  

- Partition
- Group
- Divides physical cluster into virtual ones

Not every object in a cluster lives in a namespace.
- StorageClass
- PersistentVolume
- IngressClass

In [8]:
%%bash
kubectl get pods

No resources found in default namespace.


In [9]:
%%bash
kubectl get namespaces

NAME              STATUS   AGE
default           Active   7m26s
kube-node-lease   Active   7m26s
kube-public       Active   7m26s
kube-system       Active   7m26s


In [10]:
%%bash
# kubectl get pods --all-namespaces
kubectl get pods -A

NAMESPACE     NAME                                      READY   STATUS      RESTARTS   AGE
kube-system   coredns-c4dbffb5f-t85tb                   1/1     Running     0          8m5s
kube-system   helm-install-traefik-5hzdb                0/1     Completed   0          7m59s
kube-system   helm-install-traefik-crd-b8rd2            0/1     Completed   0          7m59s
kube-system   local-path-provisioner-5c4dc5d66d-7bhbz   1/1     Running     0          8m5s
kube-system   metrics-server-786d997795-tvgpg           1/1     Running     0          8m2s
kube-system   svclb-traefik-4fd1d3b4-45zr7              2/2     Running     0          7m26s
kube-system   svclb-traefik-4fd1d3b4-64whb              2/2     Running     0          7m26s
kube-system   svclb-traefik-4fd1d3b4-cw2xr              2/2     Running     0          7m26s
kube-system   svclb-traefik-4fd1d3b4-ql9fm              2/2     Running     0          7m26s
kube-system   traefik-59449f8f96-fg68k                  1/1     Running    

### Creating a namespace


In [11]:
%%bash
kubectl create namespace k8s-ns

namespace/k8s-ns created


In [12]:
%%bash
kubectl get namespaces

NAME              STATUS   AGE
default           Active   8m46s
k8s-ns            Active   10s
kube-node-lease   Active   8m46s
kube-public       Active   8m46s
kube-system       Active   8m46s


## Workloads
- Pods (runs containers)
- Jobs
- CronJobs

In [ ]:
%%bash
kubectl get pods,jobs,cronjobs -A -o wide

### Pods

*What do we lose by running bare pods?*

A Pod is the smallest deployable unit in Kubernetes. It usually runs one container but can run more than one, and a typical multi-container setup is a sidecar.

![pod](images/pod.png)

#### Creating a POD from the CLI

In [13]:
%%bash
kubectl run k8s-pod --image=nginx -n k8s-ns

pod/k8s-pod created


In [15]:
%%bash
kubectl get pods -n k8s-ns

NAME      READY   STATUS    RESTARTS   AGE
k8s-pod   1/1     Running   0          86s


#### The describe command

In [16]:
%%bash
kubectl describe pod k8s-pod -n k8s-ns

Name:             k8s-pod
Namespace:        k8s-ns
Priority:         0
Service Account:  default
Node:             k3d-objectscluster-agent-2/172.19.0.3
Start Time:       Thu, 29 Oct 2026 16:57:08 +0200
Labels:           run=k8s-pod
Annotations:      <none>
Status:           Running
IP:               10.42.2.4
IPs:
  IP:  10.42.2.4
Containers:
  k8s-pod:
    Container ID:   containerd://d7d1aaca77c5de0a51952c5a2a049f7a9217d0fcdca264a6ad65d269a31b3a33
    Image:          nginx
    Image ID:       docker.io/library/nginx@sha256:600f7cd45e27aa13e9304c834ab7c1b0ff39df85ee1dcc5087955455d77bc83b
    Port:           <none>
    Host Port:      <none>
    State:          Running
      Started:      Thu, 29 Oct 2026 16:57:20 +0200
    Ready:          True
    Restart Count:  0
    Environment:    <none>
    Mounts:
      /var/run/secrets/kubernetes.io/serviceaccount from kube-api-access-xtbjb (ro)
Conditions:
  Type                        Status
  PodReadyToStartContainers   True 
  Initialized 

In [17]:
%%bash
kubectl delete pod --all -n k8s-ns
kubectl get pods -n k8s-ns

pod "k8s-pod" deleted from k8s-ns namespace


No resources found in k8s-ns namespace.


#### Creating a POD from the manifest file

In [18]:
%%bash
kubectl apply -f manifest/1-create_pod.yml

pod/k8s-pod created


In [19]:
%%bash
kubectl get pods -n k8s-ns

NAME      READY   STATUS    RESTARTS   AGE
k8s-pod   1/1     Running   0          3s


In [20]:
%%bash
kubectl delete -f manifest/1-create_pod.yml

pod "k8s-pod" deleted from k8s-ns namespace


### Deployments

A Deployment declares your intent: the desired state of your app, such as which image to run and how many identical Pod copies. 

![deployment](images/deployment.png)

#### Creating a DEPLOYMENT from the CLI

In [21]:
%%bash
kubectl create deployment k8s-deployment --image=nginx -n k8s-ns

deployment.apps/k8s-deployment created


In [22]:
%%bash 
kubectl get deployments -n k8s-ns

NAME             READY   UP-TO-DATE   AVAILABLE   AGE
k8s-deployment   1/1     1            1           97s


##### Creating a REPLICASET from the CLI

> DEPLOYMENTS are SCALABLE

In [25]:
%%bash
kubectl scale deploy k8s-deployment --replicas 3 -n k8s-ns

deployment.apps/k8s-deployment scaled


> DEPLOYMENTS brings AVAILABILITY

In [27]:
%%bash
# kubectl delete pod [NAME] -n k8s-ns
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
echo "Deleting pod: $POD"
kubectl delete pod $POD -n k8s-ns

Deleting pod: k8s-deployment-df7cc5847-7h6g9
pod "k8s-deployment-df7cc5847-7h6g9" deleted from k8s-ns namespace


In [28]:
%%bash
kubectl delete pod --all -n k8s-ns

pod "k8s-deployment-df7cc5847-4jkdh" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-pwsj9" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-vdzkf" deleted from k8s-ns namespace


In [29]:
%%bash
kubectl delete deployment k8s-deployment -n k8s-ns

deployment.apps "k8s-deployment" deleted from k8s-ns namespace


#### Creating a DEPLOYMENT using a manifest file

In [30]:
%%bash
kubectl apply -f manifest/2-create_deployment.yml

deployment.apps/k8s-deployment created


## Labels

Labels are the magic glue of Kubernetes: simple key-value tags you attach to objects like Pods. Selectors use them to find and group those objects, which is how a Deployment knows which Pods it owns and a Service knows which Pods to send traffic to.

![labels](images/labels.png)

In [31]:
%%bash
kubectl get pods -n k8s-ns --show-labels

NAME                             READY   STATUS    RESTARTS   AGE     LABELS
k8s-deployment-df7cc5847-dlqqh   1/1     Running   0          2m25s   app=k8s-deployment,pod-template-hash=df7cc5847
k8s-deployment-df7cc5847-ttlzr   1/1     Running   0          2m25s   app=k8s-deployment,pod-template-hash=df7cc5847
k8s-deployment-df7cc5847-xsqdm   1/1     Running   0          2m25s   app=k8s-deployment,pod-template-hash=df7cc5847


In [32]:
%%bash
# Get all pods and replicasets across all namespaces with the label app=k8s-deployment
kubectl get pods,replicaset -A -l app=k8s-deployment

NAMESPACE   NAME                                 READY   STATUS    RESTARTS   AGE
k8s-ns      pod/k8s-deployment-df7cc5847-dlqqh   1/1     Running   0          3m6s
k8s-ns      pod/k8s-deployment-df7cc5847-ttlzr   1/1     Running   0          3m6s
k8s-ns      pod/k8s-deployment-df7cc5847-xsqdm   1/1     Running   0          3m6s

NAMESPACE   NAME                                       DESIRED   CURRENT   READY   AGE
k8s-ns      replicaset.apps/k8s-deployment-df7cc5847   3         3         3       3m7s


In [33]:
%%bash
# Delete all pods and replicasets across all namespaces with the label app=k8s-deployment
kubectl delete pods,replicaset -A -l app=k8s-deployment

pod "k8s-deployment-df7cc5847-dlqqh" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-ttlzr" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-xsqdm" deleted from k8s-ns namespace
replicaset.apps "k8s-deployment-df7cc5847" deleted from k8s-ns namespace


In [34]:
%%bash
# Label the pod k8s-deployment-* with environment=dev in the k8s-ns namespace
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl label pod $POD tenant=k8sforum -n k8s-ns

pod/k8s-deployment-df7cc5847-29ksw labeled


In [35]:
%%bash
POD=$(kubectl get pod -n k8s-ns -l app=k8s-deployment -o jsonpath='{.items[0].metadata.name}')
kubectl describe pod $POD -n k8s-ns

Name:             k8s-deployment-df7cc5847-29ksw
Namespace:        k8s-ns
Priority:         0
Service Account:  default
Node:             k3d-objectscluster-server-0/172.19.0.2
Start Time:       Thu, 29 Oct 2026 17:17:24 +0200
Labels:           app=k8s-deployment
                  pod-template-hash=df7cc5847
                  tenant=k8sforum
Annotations:      <none>
Status:           Running
IP:               10.42.3.10
IPs:
  IP:           10.42.3.10
Controlled By:  ReplicaSet/k8s-deployment-df7cc5847
Containers:
  nginx:
    Container ID:   containerd://98193cca8f3ee7aaa01dd7d138ef11bd6b4b0050ad47f57e823541c2efc3ad99
    Image:          nginx
    Image ID:       docker.io/library/nginx@sha256:600f7cd45e27aa13e9304c834ab7c1b0ff39df85ee1dcc5087955455d77bc83b
    Port:           <none>
    Host Port:      <none>
    State:          Running
      Started:      Thu, 29 Oct 2026 17:17:26 +0200
    Ready:          True
    Restart Count:  0
    Environment:    <none>
    Mounts:
      /var/

In [36]:
%%bash
kubectl get pods,replicaset -A -l tenant=k8sforum

NAMESPACE   NAME                                 READY   STATUS    RESTARTS   AGE
k8s-ns      pod/k8s-deployment-df7cc5847-29ksw   1/1     Running   0          119s


## Networking

### The Data Plane

A **data plane** is a path user traffic takes to reach a workload (as opposed to the control plane, which manages the cluster). 

> Pods run on a private virtual network inside the cluster. Their IP addresses are only routable from within the cluster, and they change every time a Pod is replaced.

*How can we reach pods we created?*

In this section we follow that path from outside the cluster to a Pod.

| Concept | What it gives us |
|---|---|
| Port Forwarding | A temporary tunnel from your laptop to one Pod or Service, for debugging |
| Service & Endpoints | A stable IP address and DNS name that load-balances across the Pods matching a label selector |
| LoadBalancer | An external IP that exposes a Service outside the cluster |
| Ingress & Ingress Controller | Host and path based HTTP routing for many Services behind one entry point |

![dataplane](images/data%20plane.gif)

In [38]:
%%bash
kubectl exec deploy/k8s-deployment -n k8s-ns  -- sh -c "curl http://127.0.0.1:80"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100   896  100   896    0     0  1446k      0 --:--:-- --:--:-- --:--:--  875k


<!DOCTYPE html>
<html>
<head>
<title>Welcome to nginx!</title>
<style>
html { color-scheme: light dark; }
body { width: 35em; margin: 0 auto;
font-family: Tahoma, Verdana, Arial, sans-serif; }
</style>
</head>
<body>
<h1>Welcome to nginx!</h1>
<p>If you see this page, nginx is successfully installed and working.
Further configuration is required for the web server, reverse proxy, 
API gateway, load balancer, content cache, or other features.</p>

<p>For online documentation and support please refer to
<a href="https://nginx.org/">nginx.org</a>.<br/>
To engage with the community please visit
<a href="https://community.nginx.org/">community.nginx.org</a>.<br/>
For enterprise grade support, professional services, additional 
security features and capabilities please refer to
<a href="https://f5.com/nginx">f5.com/nginx</a>.</p>

<p><em>Thank you for using nginx.</em></p>
</body>
</html>


### Port Forwarding

Port forwarding opens a <u>temporary tunnel</u> from a port inside the cluster, so you can reach a private workload for debugging without exposing it to anyone else.

In [ ]:
%%bash
kubectl port-forward -n k8s-ns deployment/k8s-deployment 8080:80

### Service

A Service gives a group of related pods a <u>stable IP address</u> and a DNS name. 


In [41]:
%%bash
kubectl get pods -A -l app=k8s-deployment -o wide

NAMESPACE   NAME                             READY   STATUS    RESTARTS   AGE     IP           NODE                          NOMINATED NODE   READINESS GATES
k8s-ns      k8s-deployment-df7cc5847-29ksw   1/1     Running   0          9m40s   10.42.3.10   k3d-objectscluster-server-0   <none>           <none>
k8s-ns      k8s-deployment-df7cc5847-2rs8d   1/1     Running   0          9m40s   10.42.0.8    k3d-objectscluster-agent-0    <none>           <none>
k8s-ns      k8s-deployment-df7cc5847-c4562   1/1     Running   0          9m40s   10.42.1.8    k3d-objectscluster-agent-1    <none>           <none>


In [42]:
%%bash
kubectl delete pod --all -n k8s-ns

pod "k8s-deployment-df7cc5847-29ksw" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-2rs8d" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-c4562" deleted from k8s-ns namespace


#### Create a SERVICE from the CLI

In [43]:
%%bash
kubectl expose deployment k8s-deployment --port=80 --target-port=8080 --name=k8s-service --type=ClusterIP -n k8s-ns

service/k8s-service exposed


In [46]:
%%bash

echo ""
kubectl get svc k8s-service -n k8s-ns -o wide

echo ""
kubectl get endpointslices -n k8s-ns -o wide


echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns



NAME          TYPE        CLUSTER-IP    EXTERNAL-IP   PORT(S)   AGE     SELECTOR
k8s-service   ClusterIP   10.43.44.47   <none>        80/TCP    2m39s   app=k8s-deployment

NAME                ADDRESSTYPE   PORTS   ENDPOINTS                         AGE
k8s-service-77vcw   IPv4          8080    10.42.0.10,10.42.2.13,10.42.1.9   2m39s

NAME                             READY   STATUS    RESTARTS   AGE   IP           NODE                         NOMINATED NODE   READINESS GATES
k8s-deployment-df7cc5847-lkf4q   1/1     Running   0          35s   10.42.1.9    k3d-objectscluster-agent-1   <none>           <none>
k8s-deployment-df7cc5847-lrhbw   1/1     Running   0          35s   10.42.2.13   k3d-objectscluster-agent-2   <none>           <none>
k8s-deployment-df7cc5847-xcg9x   1/1     Running   0          35s   10.42.0.10   k3d-objectscluster-agent-0   <none>           <none>


In [45]:
%%bash
kubectl delete pod --all -n k8s-ns

pod "k8s-deployment-df7cc5847-c6gwt" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-mbn8z" deleted from k8s-ns namespace
pod "k8s-deployment-df7cc5847-nvnm4" deleted from k8s-ns namespace


##### CoreDNS
The cluster's <u>built-in</u> DNS server. It resolves Service names to ClusterIP addresses, so pods can find each other by name.

In [47]:
%%bash
kubectl get pods -n kube-system --show-labels

NAME                                      READY   STATUS      RESTARTS   AGE   LABELS
coredns-c4dbffb5f-t85tb                   1/1     Running     0          46m   k8s-app=kube-dns,pod-template-hash=c4dbffb5f
helm-install-traefik-5hzdb                0/1     Completed   0          46m   batch.kubernetes.io/controller-uid=6ab61204-be07-4c54-afd9-b724c910d821,batch.kubernetes.io/job-name=helm-install-traefik,controller-uid=6ab61204-be07-4c54-afd9-b724c910d821,helmcharts.helm.cattle.io/chart=traefik,job-name=helm-install-traefik
helm-install-traefik-crd-b8rd2            0/1     Completed   0          46m   batch.kubernetes.io/controller-uid=45931a05-dbb1-474a-853c-04849d656289,batch.kubernetes.io/job-name=helm-install-traefik-crd,controller-uid=45931a05-dbb1-474a-853c-04849d656289,helmcharts.helm.cattle.io/chart=traefik-crd,job-name=helm-install-traefik-crd
local-path-provisioner-5c4dc5d66d-7bhbz   1/1     Running     0          46m   app=local-path-provisioner,pod-template-hash=5c4dc5d6

In [ ]:
%%bash
# Describe all pods with the label k8s-app=kube-dns in the kube-system namespace
kubectl describe pod -l k8s-app=kube-dns -n kube-system

#### Open a cluster port

In [48]:
%%bash
# Edit the k3d cluster to add a port mapping for the load balancer
k3d cluster edit objectscluster --port-add "8081:8081@loadbalancer"

INFO[0000] portmapping '8081:8081' targets the loadbalancer: defaulting to [servers:*:proxy agents:*:proxy] 
INFO[0000] Renaming existing node k3d-objectscluster-serverlb to k3d-objectscluster-serverlb-auubn... 
INFO[0000] Creating new node k3d-objectscluster-serverlb... 
INFO[0000] Stopping existing node k3d-objectscluster-serverlb-auubn... 
INFO[0010] Starting new node k3d-objectscluster-serverlb... 
INFO[0010] Starting node 'k3d-objectscluster-serverlb'  
INFO[0017] Deleting old node k3d-objectscluster-serverlb-auubn... 
INFO[0017] Successfully updated objectscluster          


##### The LoadBalancer Service

A Service type that exposes pods outside the cluster through an external load balancer, giving them a <u>single public IP</u> that spreads traffic across the pods.

In [49]:
%%bash
# Delete the Kubernetes service named k8s-service in the k8s-ns namespace
kubectl delete svc k8s-service -n k8s-ns

service "k8s-service" deleted from k8s-ns namespace


In [50]:
%%bash
# Expose the k8s-deployment as a service of type LoadBalancer in the k8s-ns namespace
kubectl expose deployment k8s-deployment \
  --name=k8s-service \
  --type=LoadBalancer \
  --port=8081 \
  --target-port=80 \
  -n k8s-ns


service/k8s-service exposed


In [51]:
%%bash
# Get the details of the Kubernetes service, endpoints, and pods for the k8s-deployment in the k8s-ns namespace
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpointslices -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

NAME          TYPE           CLUSTER-IP     EXTERNAL-IP                                   PORT(S)          AGE
k8s-service   LoadBalancer   10.43.125.63   172.19.0.2,172.19.0.3,172.19.0.4,172.19.0.5   8081:31233/TCP   50s

NAME                ADDRESSTYPE   PORTS   ENDPOINTS                         AGE
k8s-service-ccgwv   IPv4          80      10.42.0.10,10.42.1.9,10.42.2.13   51s

NAME                             READY   STATUS    RESTARTS   AGE     IP           NODE                         NOMINATED NODE   READINESS GATES
k8s-deployment-df7cc5847-lkf4q   1/1     Running   0          4m16s   10.42.1.9    k3d-objectscluster-agent-1   <none>           <none>
k8s-deployment-df7cc5847-lrhbw   1/1     Running   0          4m16s   10.42.2.13   k3d-objectscluster-agent-2   <none>           <none>
k8s-deployment-df7cc5847-xcg9x   1/1     Running   0          4m16s   10.42.0.10   k3d-objectscluster-agent-0   <none>           <none>


#### Create a SERVICE from a manifest file

In [52]:
%%bash
kubectl delete svc k8s-service -n k8s-ns

service "k8s-service" deleted from k8s-ns namespace


In [ ]:
%%bash
kubectl apply -f manifest/3-create_service.yml

In [ ]:
%%bash
echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns

In [ ]:
%%bash
kubectl delete -f manifest/3-create_service.yml

In [53]:
%%bash
k3d cluster edit objectscluster --port-delete "8081:8081@loadbalancer"

INFO[0000] portmapping '8081:8081' targets the loadbalancer: defaulting to [servers:*:proxy agents:*:proxy] 
INFO[0000] Renaming existing node k3d-objectscluster-serverlb to k3d-objectscluster-serverlb-cVAie... 
INFO[0000] Creating new node k3d-objectscluster-serverlb... 
INFO[0000] Stopping existing node k3d-objectscluster-serverlb-cVAie... 
INFO[0010] Starting new node k3d-objectscluster-serverlb... 
INFO[0010] Starting node 'k3d-objectscluster-serverlb'  
INFO[0017] Deleting old node k3d-objectscluster-serverlb-cVAie... 
INFO[0017] Successfully updated objectscluster          


### Ingress and Ingress Controller

#### Ingress (Intent)
Holds the routing rules (host and path to Service). The controller reads them and applies them at layer 7 (HTTP/HTTPS)

#### Ingress Controller (Infrastructure)
Runs as pods in the cluster and is usually exposed through a single LoadBalancer Service. All external traffic enters through that one IP

In [54]:
%%bash
k3d cluster edit objectscluster --port-add "8082:80@loadbalancer"

INFO[0000] portmapping '8082:80' targets the loadbalancer: defaulting to [servers:*:proxy agents:*:proxy] 
INFO[0000] Renaming existing node k3d-objectscluster-serverlb to k3d-objectscluster-serverlb-CBIFW... 
INFO[0000] Creating new node k3d-objectscluster-serverlb... 
INFO[0000] Stopping existing node k3d-objectscluster-serverlb-CBIFW... 
INFO[0010] Starting new node k3d-objectscluster-serverlb... 
INFO[0010] Starting node 'k3d-objectscluster-serverlb'  
INFO[0016] Deleting old node k3d-objectscluster-serverlb-CBIFW... 
INFO[0016] Successfully updated objectscluster          


In [55]:
%%bash
kubectl apply -f manifest/4-create_ingress.yml

deployment.apps/k8s-deployment unchanged
service/k8s-service created
ingress.networking.k8s.io/k8s-ingress created


In [56]:
%%bash
echo ""
kubectl get svc k8s-service -n k8s-ns

echo ""
kubectl get endpoints k8s-service -n k8s-ns

echo ""
kubectl get ingress -n k8s-ns

echo ""
kubectl get pods -l app=k8s-deployment -o wide -n k8s-ns


NAME          TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)    AGE
k8s-service   ClusterIP   10.43.114.184   <none>        8082/TCP   4s



NAME          ENDPOINTS                                  AGE
k8s-service   10.42.0.10:80,10.42.1.9:80,10.42.2.13:80   4s

NAME          CLASS     HOSTS       ADDRESS                                       PORTS   AGE
k8s-ingress   traefik   k8s.local   172.19.0.2,172.19.0.3,172.19.0.4,172.19.0.5   80      4s

NAME                             READY   STATUS    RESTARTS   AGE     IP           NODE                         NOMINATED NODE   READINESS GATES
k8s-deployment-df7cc5847-lkf4q   1/1     Running   0          7m40s   10.42.1.9    k3d-objectscluster-agent-1   <none>           <none>
k8s-deployment-df7cc5847-lrhbw   1/1     Running   0          7m40s   10.42.2.13   k3d-objectscluster-agent-2   <none>           <none>
k8s-deployment-df7cc5847-xcg9x   1/1     Running   0          7m40s   10.42.0.10   k3d-objectscluster-agent-0   <none>           <none>


##### Update hostnames

In [57]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    current = f.read()

missing = [host for host in hosts if host not in current]
for host in hosts:
    if host not in missing:
        print(f"Already present: {host}")

if missing:
    password = getpass.getpass("sudo password: ")
    for host in missing:
        entry = f"127.0.0.1  {host}\n"
        result = subprocess.run(
            ["sudo", "-S", "tee", "-a", "/etc/hosts"],
            input=f"{password}\n{entry}",
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"Added: {host}")
        else:
            print(f"Failed: {host} - {result.stderr.strip()}")

Added: k8s.local


## Cleanup

### Delete Cluster

In [58]:
%%bash
export PATH="/opt/homebrew/bin:/usr/local/bin:$PATH"
# WINDOWS (Git Bash / WSL kernel): swap the line above for your install folder, e.g.
#   export PATH="/c/Program Files/Docker/Docker/resources/bin:$PATH"
# Delete the entire cluster - removes all Docker containers and volumes
k3d cluster delete objectscluster

INFO[0000] Deleting cluster 'objectscluster'            
INFO[0004] Deleting cluster network 'k3d-objectscluster' 
INFO[0005] Deleting 1 attached volumes...               
INFO[0005] Removing cluster details from default kubeconfig... 
INFO[0005] Removing standalone kubeconfig file (if there is one)... 
INFO[0005] Successfully deleted cluster objectscluster! 


### Reclaim Disk Space

In [ ]:
%%bash
docker system df
echo ""
echo "=== Dangling volumes ==="
docker volume ls --filter dangling=true
echo ""
docker volume prune -f

### Delete Host Name

**macOS / Linux**

In [59]:
import subprocess
import getpass

hosts = [
    "k8s.local"
]

with open("/etc/hosts", "r") as f:
    lines = f.readlines()

present = [host for host in hosts if any(host in line.split() for line in lines)]
for host in hosts:
    if host not in present:
        print(f"Not present: {host}")

if present:
    password = getpass.getpass("sudo password: ")
    kept = [line for line in lines if not any(host in line.split() for host in present)]
    result = subprocess.run(
        ["sudo", "-S", "tee", "/etc/hosts"],
        input=f"{password}\n{''.join(kept)}",
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        for host in present:
            print(f"Removed: {host}")
    else:
        print(f"Failed: {result.stderr.strip()}")

Removed: k8s.local
